# 📊 Phase 1: Data Preparation

We build TWO datasets in this notebook:
1. **SFT dataset** — (instruction, correct JSON output) pairs for supervised fine-tuning
2. **DPO dataset** — (prompt, chosen_output, rejected_output) triples for preference tuning

Source: `argilla/distilabel-intel-orca-dpo-pairs` — already has chosen/rejected structure,
we'll filter and reformat for JSON extraction task.

In [1]:
# ── Global config (copy from 00_setup) ───────────────────────────────────────
BASE_MODEL     = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN    = 1024
SEED           = 42
import os, json, random
os.makedirs('./data', exist_ok=True)
random.seed(SEED)

## Part A — Build SFT Dataset (JSON Extraction)

We create a synthetic but realistic JSON extraction dataset.
Each example: messy unstructured text → clean structured JSON.
This is the clearest way to demonstrate the task with measurable metrics.

In [2]:
# ── Define JSON extraction schemas & templates ────────────────────────────────
# We use 4 real-world schemas to keep variety high

SCHEMAS = {
    "person": {
        "fields": ["name", "age", "email", "city"],
        "description": "Extract person contact details"
    },
    "product": {
        "fields": ["product_name", "price", "category", "in_stock"],
        "description": "Extract product listing details"
    },
    "event": {
        "fields": ["event_name", "date", "location", "organizer"],
        "description": "Extract event details"
    },
    "invoice": {
        "fields": ["invoice_number", "amount", "due_date", "client_name"],
        "description": "Extract invoice details"
    }
}

print('Schemas defined:', list(SCHEMAS.keys()))

Schemas defined: ['person', 'product', 'event', 'invoice']


In [3]:
# ── Generate synthetic training examples ──────────────────────────────────────
import random
from datasets import Dataset

PERSON_DATA = [
    {"name": "Alice Johnson", "age": 32, "email": "alice@example.com", "city": "Austin"},
    {"name": "Bob Chen", "age": 45, "email": "bchen@corp.io", "city": "Seattle"},
    {"name": "Maria Garcia", "age": 28, "email": "m.garcia@mail.net", "city": "Miami"},
    {"name": "James Patel", "age": 55, "email": "jpatel@biz.com", "city": "Chicago"},
    {"name": "Sara Kim", "age": 23, "email": "sara.kim@web.dev", "city": "San Francisco"},
]
PRODUCT_DATA = [
    {"product_name": "Wireless Headphones Pro", "price": 129.99, "category": "Electronics", "in_stock": True},
    {"product_name": "Yoga Mat Premium", "price": 45.00, "category": "Sports", "in_stock": True},
    {"product_name": "Coffee Grinder X200", "price": 89.50, "category": "Kitchen", "in_stock": False},
    {"product_name": "Running Shoes Air", "price": 210.00, "category": "Footwear", "in_stock": True},
]
EVENT_DATA = [
    {"event_name": "AI Summit 2025", "date": "2025-03-15", "location": "San Francisco", "organizer": "TechConf Inc."},
    {"event_name": "Startup Expo", "date": "2025-04-22", "location": "New York", "organizer": "Venture Hub"},
    {"event_name": "Design Week", "date": "2025-05-10", "location": "London", "organizer": "Creative Guild"},
]
INVOICE_DATA = [
    {"invoice_number": "INV-2024-001", "amount": 4500.00, "due_date": "2024-12-31", "client_name": "Acme Corp"},
    {"invoice_number": "INV-2024-055", "amount": 1200.50, "due_date": "2024-11-15", "client_name": "Beta LLC"},
    {"invoice_number": "INV-2025-003", "amount": 750.00, "due_date": "2025-01-20", "client_name": "Gamma Inc"},
]

def make_messy_person(p):
    templates = [
        f"Hey, I wanted to follow up with {p['name']}. They're {p['age']} years old and based in {p['city']}. Best way to reach them is {p['email']}.",
        f"{p['name']} ({p['email']}) is a {p['age']}-year-old professional located in {p['city']}.",
        f"Please add {p['name']} to the mailing list. City: {p['city']}, Age: {p['age']}, Email address is {p['email']}.",
        f"Contact info dump — Name: {p['name']}, residing in {p['city']}, aged {p['age']}, reachable at {p['email']}.",
    ]
    return random.choice(templates)

def make_messy_product(pr):
    stock_str = "currently available" if pr['in_stock'] else "out of stock right now"
    templates = [
        f"We have the {pr['product_name']} listed under {pr['category']} for ${pr['price']}. It is {stock_str}.",
        f"Product: {pr['product_name']} — Category is {pr['category']}, priced at ${pr['price']}, and it's {stock_str}.",
        f"{pr['product_name']} (${pr['price']}) falls under our {pr['category']} section. Stock status: {stock_str}.",
    ]
    return random.choice(templates)

def make_messy_event(e):
    templates = [
        f"{e['event_name']} is scheduled for {e['date']} in {e['location']}, organized by {e['organizer']}.",
        f"Don't miss {e['event_name']}! It's happening on {e['date']} at {e['location']}. {e['organizer']} is putting it together.",
        f"Save the date: {e['event_name']}, {e['location']}, {e['date']}. Brought to you by {e['organizer']}.",
    ]
    return random.choice(templates)

def make_messy_invoice(inv):
    templates = [
        f"Invoice {inv['invoice_number']} for {inv['client_name']} is due by {inv['due_date']}. Total amount: ${inv['amount']}.",
        f"Please process payment of ${inv['amount']} for {inv['client_name']}. Invoice ref: {inv['invoice_number']}, due {inv['due_date']}.",
        f"{inv['client_name']} owes ${inv['amount']} per invoice {inv['invoice_number']}. Due date is {inv['due_date']}.",
    ]
    return random.choice(templates)

SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def make_instruction(schema_name, fields):
    return f"Extract the following fields from the text and return as JSON: {', '.join(fields)}."

sft_examples = []

# Generate examples for each schema, repeat to hit ~2000 examples
REPEATS = 125

for _ in range(REPEATS):
    for p in PERSON_DATA:
        text = make_messy_person(p)
        instruction = make_instruction("person", SCHEMAS["person"]["fields"])
        sft_examples.append({
            "schema": "person",
            "input_text": text,
            "instruction": instruction,
            "output": json.dumps({k: p[k] for k in SCHEMAS["person"]["fields"]}),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{instruction}\n\nText: {text}"},
                {"role": "assistant", "content": json.dumps({k: p[k] for k in SCHEMAS["person"]["fields"]})}
            ]
        })
    for pr in PRODUCT_DATA:
        text = make_messy_product(pr)
        instruction = make_instruction("product", SCHEMAS["product"]["fields"])
        sft_examples.append({
            "schema": "product",
            "input_text": text,
            "instruction": instruction,
            "output": json.dumps({k: pr[k] for k in SCHEMAS["product"]["fields"]}),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{instruction}\n\nText: {text}"},
                {"role": "assistant", "content": json.dumps({k: pr[k] for k in SCHEMAS["product"]["fields"]})}
            ]
        })
    for e in EVENT_DATA:
        text = make_messy_event(e)
        instruction = make_instruction("event", SCHEMAS["event"]["fields"])
        sft_examples.append({
            "schema": "event",
            "input_text": text,
            "instruction": instruction,
            "output": json.dumps({k: e[k] for k in SCHEMAS["event"]["fields"]}),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{instruction}\n\nText: {text}"},
                {"role": "assistant", "content": json.dumps({k: e[k] for k in SCHEMAS["event"]["fields"]})}
            ]
        })
    for inv in INVOICE_DATA:
        text = make_messy_invoice(inv)
        instruction = make_instruction("invoice", SCHEMAS["invoice"]["fields"])
        sft_examples.append({
            "schema": "invoice",
            "input_text": text,
            "instruction": instruction,
            "output": json.dumps({k: inv[k] for k in SCHEMAS["invoice"]["fields"]}),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{instruction}\n\nText: {text}"},
                {"role": "assistant", "content": json.dumps({k: inv[k] for k in SCHEMAS["invoice"]["fields"]})}
            ]
        })

random.shuffle(sft_examples)
print(f'Total SFT examples generated: {len(sft_examples)}')

Total SFT examples generated: 1875


In [4]:
# ── Train / Test split ────────────────────────────────────────────────────────
split_idx = int(len(sft_examples) * 0.9)
train_sft = sft_examples[:split_idx]
test_sft  = sft_examples[split_idx:]

print(f'Train: {len(train_sft)} | Test: {len(test_sft)}')

# Save to disk
with open('./data/train_sft.json', 'w') as f:
    json.dump(train_sft, f, indent=2)
with open('./data/test_sft.json', 'w') as f:
    json.dump(test_sft, f, indent=2)

print('✅ Saved train_sft.json and test_sft.json')

Train: 1687 | Test: 188
✅ Saved train_sft.json and test_sft.json


## Part B — Build DPO Dataset (Preference Pairs)

For DPO we need (prompt, chosen, rejected) triples.
- **chosen**: correct JSON output
- **rejected**: intentionally flawed output (invalid JSON, missing fields, wrong values)

In [5]:
# ── Generate DPO preference pairs ─────────────────────────────────────────────

def make_rejected_output(correct_json_str, schema_name):
    """Create intentionally bad outputs of different failure types."""
    correct = json.loads(correct_json_str)
    failure_type = random.choice(['invalid_json', 'missing_field', 'extra_text', 'wrong_format'])
    
    if failure_type == 'invalid_json':
        # Malformed JSON - most common real failure
        return correct_json_str.replace('"', "'").replace('{', '{')
    elif failure_type == 'missing_field':
        # Drop a random field
        keys = list(correct.keys())
        if len(keys) > 1:
            drop = random.choice(keys)
            reduced = {k: v for k, v in correct.items() if k != drop}
            return json.dumps(reduced)
        return "I couldn't find all the required information in the text."
    elif failure_type == 'extra_text':
        # Model adds explanation around JSON (common failure)
        return f"Here is the extracted JSON:\n```json\n{correct_json_str}\n```"
    else:
        # Nested wrong structure
        return json.dumps({"result": correct, "status": "extracted"})

dpo_examples = []
# Use first 1500 train examples as DPO source
for ex in train_sft[:1500]:
    prompt = f"{ex['instruction']}\n\nText: {ex['input_text']}"
    chosen = ex['output']
    rejected = make_rejected_output(ex['output'], ex['schema'])
    dpo_examples.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
        "schema": ex['schema']
    })

random.shuffle(dpo_examples)
split_dpo = int(len(dpo_examples) * 0.9)
train_dpo = dpo_examples[:split_dpo]
test_dpo  = dpo_examples[split_dpo:]

with open('./data/train_dpo.json', 'w') as f:
    json.dump(train_dpo, f, indent=2)
with open('./data/test_dpo.json', 'w') as f:
    json.dump(test_dpo, f, indent=2)

print(f'DPO train: {len(train_dpo)} | DPO test: {len(test_dpo)}')
print('✅ Saved train_dpo.json and test_dpo.json')

DPO train: 1350 | DPO test: 150
✅ Saved train_dpo.json and test_dpo.json


In [6]:
# ── Sanity check: inspect a few examples ──────────────────────────────────────
import pprint
print('=== SFT Example ===')
pprint.pprint(train_sft[0], width=100)
print()
print('=== DPO Example ===')
pprint.pprint(train_dpo[0], width=100)

=== SFT Example ===
{'input_text': 'We have the Yoga Mat Premium listed under Sports for $45.0. It is currently '
               'available.',
 'instruction': 'Extract the following fields from the text and return as JSON: product_name, '
                'price, category, in_stock.',
 'messages': [{'content': 'You are a precise JSON extraction assistant. \n'
                          'Given unstructured text, extract the requested fields and return ONLY '
                          'valid JSON.\n'
                          'Do not include any explanation, markdown, or extra text. Output only '
                          'the JSON object.',
               'role': 'system'},
              {'content': 'Extract the following fields from the text and return as JSON: '
                          'product_name, price, category, in_stock.\n'
                          '\n'
                          'Text: We have the Yoga Mat Premium listed under Sports for $45.0. It is '
                         